In [1]:
import pandas as pd
import os

os.getcwd()

'/Users/degn400/Git_Repos/DancePartner/vignettes'

In [2]:
import re

def __get_ome_df(ome_path: str, delim: str = ",", ome_type: str = None):
    '''
    Pull an omes files, and parse the file to be a pandas dataframe 

    Parameters
    ----------
    omes_path
        The path to the omes file formatted with the first column as the ID and the second as the Synonyms
    
    delim
        The delimiter for the file. Default is a ',' for a csv. 

    ome_type
        An optional identifier for the ome type
    '''

    # Read the ome csv
    ome = pd.read_csv(ome_path, sep = delim)

    # Instantiate the ome dictionary
    ome_dict = {}

    for row in range(len(ome)):
        
        terms = str(ome["Synonyms"][row]).split("; ")
        terms = [re.sub(r'[^a-zA-Z0-9]', '', term.strip().lower()) for term in terms]

        if not isinstance(terms, list):
            terms = list(terms)

        for do_not_use in ["", "nan"]:
            if do_not_use in terms:
                terms.remove(do_not_use)

        # Only return if there is one match
        if len(terms) > 0:
            ome_dict[ome.iloc[row, 0]] = list(set(terms))

    # Make dictionary
    df = pd.DataFrame(ome_dict.items()).explode(1).rename({0: "ID", 1: "Synonym"}, axis = 1)
    if ome_type is not None:
        df["Type"] = ome_type

    return df

In [3]:
lipidome = __get_ome_df("../omes/LipidMaps_Lipidome.csv", ome_type = "lipid")
metabolome = __get_ome_df("../omes/CHEBI_metabolome.txt", "\t", "metabolite")
proteome = __get_ome_df("../../edge_weighting_truthdata/omes/UP000000803_proteome.txt", "\t", "gene product")
genome = __get_ome_df("../../edge_weighting_truthdata/omes/7227_genome.txt", "\t", "gene product")

In [4]:
ome_data = pd.concat([lipidome, metabolome, proteome, genome])

# Remove terms that do no fit the requirements
stopwords = pd.read_csv(os.path.join("../omes/stop_words_english.txt"))["stopwords"].tolist()

# Determine all terms that meet requirements
unique_terms = list(set(ome_data["Synonym"].to_list()))
cleaned_terms = [term for term in unique_terms if len(term) >= 3 and len(term) <= 50 and term not in stopwords]

# Filter ome data down to cleaned terms 
ome_data = ome_data[ome_data["Synonym"].isin(cleaned_terms)].reset_index(drop = True)

In [10]:
Synonym_Groups = ome_data.groupby("Synonym").agg({"ID": list, "Type": list}).reset_index()
Synonym_Groups["SynGroup"] = ["SynGroup" + str(x) for x in range(len(Synonym_Groups))]
Synonym_Groups

,Synonym,ID,Type,SynGroup
0,00xf3,[CHEBI:53456],[metabolite],SynGroup0
1,01289,"[A0A0B4K7H1, A0A0B4K819, A1Z6P2, E1JGY5, E1JGY...","[gene product, gene product, gene product, gen...",SynGroup1
2,0129,[CHEBI:73908],[metabolite],SynGroup2
3,03659,[A0A0B4KEG2],[gene product],SynGroup3
4,04053,[Q9VNR6],[gene product],SynGroup4
...,...,...,...,...
214349,zzzz7111417icosatetraenoicacid,[LMFA01031069],[lipid],SynGroup214349
214350,zzzzeicosa8111417tetraenoate,[CHEBI:71563],[metabolite],SynGroup214350
214351,zzzzeicosa8111417tetraenoicacid,[CHEBI:71488],[metabolite],SynGroup214351
214352,zzzzicosa8111417tetraenoate,[CHEBI:71563],[metabolite],SynGroup214352


In [11]:
ID_Groups = ome_data.groupby("ID").agg({"Synonym": list, "Type": list}).reset_index()
ID_Groups["IDGroup"] = ["IDGroup" + str(x) for x in range(len(ID_Groups))]
ID_Groups

,ID,Synonym,Type,IDGroup
0,A0A021WW32,[verthandi],[gene product],IDGroup0
1,A0A021WZA4,"[nakca2exchangeprotein5, sodiumpotassiumcalciu...","[gene product, gene product, gene product]",IDGroup1
2,A0A023GPJ3,[chronophage],[gene product],IDGroup2
3,A0A023GPK8,"[friendofechinoid, echinoid]","[gene product, gene product]",IDGroup3
4,A0A023GPM5,"[dagkinase, diacylglycerolkinase, 271107]","[gene product, gene product, gene product]",IDGroup4
...,...,...,...,...
126066,X2JLB5,"[ribonucleasepproteinsubunitp20, p20]","[gene product, gene product]",IDGroup126066
126067,X2JLF7,"[wing, shortwing]","[gene product, gene product]",IDGroup126067
126068,X2JLM6,"[ubiquitincarboxylterminalhydrolase, hydrolase...","[gene product, gene product, gene product]",IDGroup126068
126069,X2JLN4,[27424],[gene product],IDGroup126069


In [24]:
import networkx as nx

# Save information with similar groups
collapse = pd.merge(pd.merge(ome_data, Synonym_Groups[["Synonym", "SynGroup"]]), ID_Groups[["ID", "IDGroup"]])

# Build the network of connected groups
Supergroups = nx.Graph()
Supergroups.add_edges_from(collapse[["SynGroup", "IDGroup"]].itertuples(index=False, name=None))

# Find connected nodes
connected_components = list(nx.connected_components(Supergroups))

# Create a new supergroup mapping
supergroup_mapping = {}
for i, component in enumerate(connected_components):
    for node in component:
        supergroup_mapping[node] = i

# Apply supergroup mapping
collapse["DancePartnerID"] = collapse["SynGroup"].map(supergroup_mapping).combine_first(collapse["IDGroup"].map(supergroup_mapping))

synonym_table = collapse[["DancePartnerID", "ID", "Synonym", "Type"]].groupby("DancePartnerID").agg({"ID": list, "Synonym": list, "Type": list})

In [25]:
def __get_label_priority(ome_list: list):
    '''
    From a list of any omes, rank the order by lipid, metabolite, and gene product
    '''
    
    if "lipid" in ome_list:
        return "lipid"
    elif "metabolite" in ome_list:
        return "metabolite"
    else:
        return "gene product"

In [32]:
synonym_table["Type"] = [__get_label_priority(ome_list) for ome_list in synonym_table["Type"]]
synonym_table = synonym_table.explode(["ID", "Synonym"]).reset_index()
synonym_table

,DancePartnerID,ID,Synonym,Type
0,0,LMFA00000001,acetylenicacids,lipid
1,0,LMFA00000001,2methoxy12methyloctadec17en5ynoylanhydride,lipid
2,0,LMFA01030475,acetylenicacids,lipid
3,0,LMFA01030475,10e16heptadecadien8ynoicacid,lipid
4,0,LMFA01030475,1016heptadecadien8ynoicacide,lipid
...,...,...,...,...
267835,86151,FBgn0267595,cr45933,gene product
267836,86152,FBgn0259864,cr42433,gene product
267837,86153,FBgn0085506,cg40635,gene product
267838,86154,FBgn0259870,cr42439,gene product


In [39]:
# Load output from UniProt

uniprot = pd.read_csv("../../edge_weighting_truthdata/drosophila/7227_uniprot.txt", sep = "\t")[["ID1", "ID2"]]

uniprot = pd.merge(uniprot, synonym_table.rename(columns = {"ID":"ID1", "Synonym":"Synonym1", "Type":"Type1", "DancePartnerID":"DancePartnerID1"}))
uniprot = pd.merge(uniprot, synonym_table.rename(columns = {"ID":"ID2", "Synonym":"Synonym2", "Type":"Type2", "DancePartnerID":"DancePartnerID2"}))

uncollapsed = uniprot[["DancePartnerID1", "DancePartnerID2", "Type1", "Type2"]].drop_duplicates().reset_index(drop = True)